# 🧬 CancerBindNet — AI Cancer Drug Discovery

This notebook runs the full CancerBindNet pipeline:
1. Load BindingDB binding affinity data
2. Filter for cancer targets (EGFR, IDH1, VEGFR, etc.)
3. Train Neural Network + Random Forest to predict IC50
4. Score new drug candidates

**Runtime:** Runtime → Change runtime type → T4 GPU

## Step 1 — Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Unzip project
import zipfile
with zipfile.ZipFile('/content/drive/MyDrive/cdd_bindingdb.zip', 'r') as z:
    z.extractall('/content/')
print('Project ready!')

## Step 2 — Install Dependencies

In [ ]:
!pip install rdkit torch pandas scikit-learn scipy tqdm -q
print('Dependencies installed!')

## Step 3 — Copy BindingDB to Local Storage

Copying from Drive to local disk prevents timeout issues with the large file.

In [ ]:
import shutil, os
print('Copying BindingDB zip to local storage...')
shutil.copy(
    '/content/drive/MyDrive/BindingDB_All_202605_tsv.zip',
    '/content/BindingDB_All_202605_tsv.zip'
)
size = os.path.getsize('/content/BindingDB_All_202605_tsv.zip') / 1e6
print(f'Done! File size: {size:.0f} MB')

## Step 4 — Process BindingDB Data

In [ ]:
%cd /content/cdd_bindingdb
!python data/prepare_bindingdb.py \
    --input /content/BindingDB_All_202605_tsv.zip \
    --max-rows 500000

## Step 5 — Train Model on EGFR

In [ ]:
!python scripts/train.py --target EGFR --epochs 50

## Step 6 — Predict on Known EGFR Inhibitors

In [ ]:
# Test with approved EGFR drugs + negative control
molecules = """COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1
C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1
CC(=O)Oc1ccccc1C(=O)O
CN1CCC[C@H]1c1cccnc1
CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C
Clc1ccc2[nH]c3ccccc3c2c1
CCN(CC)CCNC(=O)c1ccc(N)cc1
Fc1ccc(Cn2nnc3ccccc32)cc1
O=C(O)c1ccccc1O
c1ccc2[nH]ccc2c1""".strip()

with open('test_molecules.txt', 'w') as f:
    f.write(molecules)

!python scripts/predict.py --smiles-file test_molecules.txt

## Step 7 — Save Model to Drive

In [ ]:
import glob, shutil, os
runs = sorted(glob.glob('/content/cdd_bindingdb/runs/*'), key=os.path.getmtime)
latest = runs[-1]
dest = f'/content/drive/MyDrive/CancerBindNet_{os.path.basename(latest)}'
shutil.copytree(latest, dest)
print(f'Model saved to Drive: {dest}')